# Delivery Time Prediction

**Tabular Regression Project** — Predict food delivery duration.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Food Delivery (Kaggle)
Target: Time taken (minutes)

## 2 · Project Overview

We predict **food delivery time** based on order and delivery partner features. This is a practical logistics regression problem — accurate delivery estimates improve customer satisfaction and operational planning.

## 3 · Learning Objectives

1. Download and preprocess delivery data from Kaggle.
2. Handle mixed feature types (numeric, categorical, geographic).
3. Engineer distance and time-based features.
4. Compare regression models for delivery time prediction.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Predict the **time taken** for food delivery given order details, restaurant location, delivery location, and delivery partner information.

## 5 · Why This Project Matters

- **Delivery time estimation** directly affects customer experience.
- Accurate predictions help optimize driver routing and dispatching.
- Food delivery is a massive and growing industry.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: gauravmalik26/food-delivery-dataset |
| **Target** | Time_taken(min) or similar delivery duration column |
| **Note** | Downloaded via kagglehub at runtime |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle food delivery dataset.
- **License**: Check Kaggle page.
- **Note**: Real-world delivery data with geographic coordinates.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [4]:
import kagglehub, glob

try:
    path = kagglehub.dataset_download('gauravmalik26/food-delivery-dataset')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR

csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
data_file = sorted(csv_files, key=os.path.getsize, reverse=True)[0]
print(f'Using: {data_file}')
df = pd.read_csv(data_file)
print(f'Loaded: {df.shape}')
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f'Sampled to: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

  0%|                                              | 0.00/1.95M [00:00<?, ?B/s]

 51%|███████████████████▍                  | 1.00M/1.95M [00:01<00:01, 635kB/s]

100%|█████████████████████████████████████| 1.95M/1.95M [00:01<00:00, 1.26MB/s]

100%|█████████████████████████████████████| 1.95M/1.95M [00:01<00:00, 1.09MB/s]

Extracting files...


Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\gauravmalik26\food-delivery-dataset\versions\1
Using: C:\Users\ahmad\.cache\kagglehub\datasets\gauravmalik26\food-delivery-dataset\versions\1\train.csv
Loaded: (45593, 20)
Columns: ['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)']


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


## 12 · Data Validation Checks

In [5]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Dtypes: {dict(df.dtypes)}')

Missing values:
ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Time_Orderd                    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Vehicle_condition              0
Type_of_order                  0
Type_of_vehicle                0
multiple_deliveries            0
Festival                       0
City                           0
Time_taken(min)                0
dtype: int64

Duplicates: 0
Dtypes: {'ID': dtype('O'), 'Delivery_person_ID': dtype('O'), 'Delivery_person_Age': dtype('O'), 'Delivery_person_Ratings': dtype('O'), 'Restaurant_latitude': dtype('float64'), 'Restaurant_longitude': dtype('float64'), 'Delivery_location_latitude': dtype('float64'), 'Delivery_location_longitude':

## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.tolist()
n_plot = min(4, len(num_cols))
if n_plot > 0:
    fig, axes = plt.subplots(1, n_plot, figsize=(4*n_plot, 4))
    if n_plot == 1: axes = [axes]
    for ax, col in zip(axes, num_cols[:n_plot]):
        df[col].dropna().hist(bins=30, ax=ax, edgecolor='black')
        ax.set_title(col[:20])
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
    plt.show()

## 14 · Target Analysis

In [7]:
# Find target column
target_candidates = [c for c in df.columns if any(kw in c.lower() for kw in ['time', 'duration', 'taken', 'deliver'])]
if target_candidates:
    TARGET = target_candidates[0]
else:
    TARGET = num_cols[-1]

# Clean target if it has text
if df[TARGET].dtype == 'object':
    import re
    df[TARGET] = df[TARGET].apply(lambda x: float(re.findall(r'[\d.]+', str(x))[0]) if re.findall(r'[\d.]+', str(x)) else np.nan)

df = df.dropna(subset=[TARGET])
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
df = df.dropna(subset=[TARGET])

print(f'Target: {TARGET}')
print(df[TARGET].describe())
print(f'Skewness: {df[TARGET].skew():.2f}')

Target: Delivery_person_ID
count    45593.000000
mean        10.487706
std          5.763282
min          1.000000
25%          5.000000
50%         10.000000
75%         15.000000
max         20.000000
Name: Delivery_person_ID, dtype: float64
Skewness: 0.00


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [8]:
from sklearn.preprocessing import LabelEncoder

# Drop high-cardinality text/ID columns
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 100 or df[col].nunique() < 2:
        df = df.drop(columns=[col])
    else:
        df[col] = df[col].fillna('Unknown')
        df[col] = LabelEncoder().fit_transform(df[col])

# Fill numeric NaN
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

Preprocessed: (45593, 17)


## 17 · Feature Engineering

In [9]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (36474, 16), Test: (9119, 16)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=5.74, MAE=4.98, R²=-0.0004

## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                               Adjusted R-Squared  R-Squared      RMSE  \
Model                                                                    
RandomForestRegressor                    0.875342   0.877338  1.965972   
BaggingRegressor                         0.860799   0.863029  2.077484   
XGBRegressor                             0.818855   0.821757  2.369897   
CatBoostRegressor                        0.791105   0.794450  2.544962   
DecisionTreeRegressor                    0.783294   0.786765  2.592103   
HistGradientBoostingRegressor            0.776017   0.779605  2.635263   
LGBMRegressor                            0.775800   0.779391  2.636540   
GradientBoostingRegressor                0.547295   0.554546  3.746489   
AdaBoostRegressor                        0.054857   0.069994  5.413344   
ElasticNet                              -0.016361  -0.000083  5.613591   
DummyRegressor                          -0.016361  -0.000083  5.613591   
LassoLarsIC                           

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models (CatBoost + LightGBM + XGBoost)

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=2.46, MAE=1.61, R²=0.8165


LightGBM: RMSE=2.09, MAE=0.91, R²=0.8679


XGBoost: RMSE=2.12, MAE=0.89, R²=0.8636


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  LightGBM              RMSE=2.09  MAE=0.91  R²=0.8679
  XGBoost               RMSE=2.12  MAE=0.89  R²=0.8636
  CatBoost              RMSE=2.46  MAE=1.61  R²=0.8165
  Baseline_LR           RMSE=5.74  MAE=4.98  R²=-0.0004

Top 3: ['LightGBM', 'XGBoost', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
LightGBM: RMSE=2.09, MAE=0.91, R²=0.8679
XGBoost: RMSE=2.12, MAE=0.89, R²=0.8636
CatBoost: RMSE=2.46, MAE=1.61, R²=0.8165


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models['CatBoost']
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- Distance and traffic conditions are typically the strongest predictors.
- Time of day and day of week affect delivery times significantly.
- These predictions can improve delivery promise accuracy and customer satisfaction.

## 26 · Limitations

- Data quality varies — real delivery data is noisy.
- No real-time traffic or weather integration.
- Geographic features may not generalize across cities.

## 27 · How to Improve

1. Add real-time traffic data.
2. Compute actual distance using Haversine formula.
3. Include weather conditions.
4. Build city-specific models.

## 28 · Production Considerations

- Real-time feature computation needed.
- Regular retraining as delivery patterns change.
- Integration with routing APIs.
- A/B testing of delivery time estimates.

## 29 · Common Mistakes

1. Not cleaning text from target column.
2. Including order ID as a feature.
3. Not computing distance from lat/lon.
4. Ignoring time-of-day effects.

## 30 · Mini Challenge

1. Compute Haversine distance and add as feature.
2. Add time-of-day buckets.
3. Compare weekday vs weekend delivery times.
4. Build a model for only the top 3 cities.

## 31 · Final Summary

- Delivery time prediction is a high-impact logistics ML problem.
- Feature engineering (distance, time features) matters more than model choice.
- Gradient-boosting models handle mixed features well.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
